In [1]:
import torch
import torchvision
import torch.nn as nn

class Net(nn.Module):
    def __init__(self, n_feature, n_hidden, n_output):
        super().__init__()
        self.hidden = torch.nn.linear(n_feature, n_hidden)
        self.relu = torch.nn.ReLU(inplace=True)
        self.out  = torch.nn.Linear(n_hidden, n_output)
        self.softmax = nn.Softmax(dim=n_output)
        self.model = nn.Sequential(self.hidden, self.relu, self.out, self.softmax)
    def forward(self, x):
        output = self.model(x)
        return output



In [4]:
class DropoutModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784,1200)
        self.dropout1 = nn.Dropout(0.5)
        self.layer2= nn.Linear(1200,1200)
        self.dropout2 = nn.Dropout(0.5)
        self.layer3= nn.Linear(1200,10)
        self.model = nn.Sequential(self.layer1, self.dropout1, self.layer2, self.dropout2, self.layer3)
    def forward(self, x):
        output= self.model(x)
        return output


In [8]:
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

train_dataset = torchvision.datasets.FashionMNIST("./data", download=True, transform=transforms.Compose([transforms.ToTensor()]))
test_dataset = torchvision.datasets.FashionMNIST("./data", download=True, train=False, transform=transforms.Compose([transforms.ToTensor()]))

In [10]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100)
labels_map = {0 : 'T-Shirt', 1 : 'Trouser', 2 : 'Pullover', 3 : 'Dress', 4 : 'Coat', 5 : 'Sandal', 6 : 'Shirt',
              7 : 'Sneaker', 8 : 'Bag', 9 : 'Ankle Boot'}

In [15]:
class FashionDNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.model =  nn.Sequential(self.fc1, self.drop, self.fc2, self.fc3)
    def forward(self, x):
        x = x.view(-1, 784)
        output = self.model(x)
        return output



In [17]:
learning_rate = 0.001;
model = FashionDNN();
criterion = nn.CrossEntropyLoss();
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate);
print(model)

FashionDNN(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (drop): Dropout(p=0.25, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): Dropout(p=0.25, inplace=False)
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [18]:
from torch.autograd import Variable
import torch.nn.functional as F

num_epochs = 5
count = 0
loss_list = []
iteration_list = []
accuracy_list = []

predictions_list = []
labels_list = []

for epoch in range(num_epochs):
    for images, labels in train_loader:
    
        train = Variable(images.view(100, 1, 28, 28))
        labels = Variable(labels)
        
        outputs = model(train)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        count += 1
    
        if not (count % 50):    
            total = 0
            correct = 0        
            for images, labels in test_loader:
                labels_list.append(labels)            
                test = Variable(images.view(100, 1, 28, 28))            
                outputs = model(test)            
                predictions = torch.max(outputs, 1)[1]
                predictions_list.append(predictions)
                correct += (predictions == labels).sum()            
                total += len(labels)
            
            accuracy = correct * 100 / total
            loss_list.append(loss.data)
            iteration_list.append(count)
            accuracy_list.append(accuracy)
        
        if not (count % 500):
            print("Iteration: {}, Loss: {}, Accuracy: {}%".format(count, loss.data, accuracy))

Iteration: 500, Loss: 0.668489933013916, Accuracy: 82.0199966430664%
Iteration: 1000, Loss: 0.4922597110271454, Accuracy: 83.13999938964844%
Iteration: 1500, Loss: 0.4482528269290924, Accuracy: 81.0999984741211%
Iteration: 2000, Loss: 0.5501816868782043, Accuracy: 82.97000122070312%
Iteration: 2500, Loss: 0.345462441444397, Accuracy: 82.8499984741211%
Iteration: 3000, Loss: 0.39268600940704346, Accuracy: 83.5%


In [2]:
class FashionCNN(nn.Module):
    def __init__(self):
        super.__init__()
        self.layer1= nn.Sequential(
            nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.layer2= nn.Sequential(
            nn.Conv2d(32,64,3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc1 = nn.Linear(64*6*6, 600)
        self.drop =nn.Dropout2d(0.25)
        self.fc2 = nn.Linear(600,120)
        self.fc3 = nn.Linear(120,10)

    def forward(self, x):
        output = self.layer1(x)
        output = self.layer2(output)
        output = output.view(output.size(0),-1)
        output = self.fc1(output)
        output = self.drop(output)
        output = self.fc2(output)
        output = self.fc3(output)
        return output
        
